## Vectorless RAG with PageIndex

Traditional RAG chunks a document, embeds the chunks, and retrieves by vector similarity. **Vectorless RAG** replaces all of that with a hierarchical, table-of-contents-like tree of the document (built by [PageIndex](https://pageindex.ai)) and lets an LLM *reason* over that tree to decide which section(s) actually answer the question — no embeddings, no vector store, no chunking.

In [1]:
import os
from dotenv import load_dotenv

# os.environ.pop("SSLKEYLOGFILE", None)  # prevent PermissionError in ssl module under sandbox environments

load_dotenv()

True

### Submitting a document to PageIndex

We download a short public paper ("Attention Is All You Need", ~15 pages) at runtime and submit it to PageIndex for processing.

In [4]:
import tempfile
import requests
from pageindex import PageIndexClient

PDF_URL = "https://arxiv.org/pdf/1706.03762"

pdf_path = os.path.join(tempfile.gettempdir(), "attention_is_all_you_need.pdf")
response = requests.get(PDF_URL)
response.raise_for_status()
with open(pdf_path, "wb") as f:
    f.write(response.content)

pi_client = PageIndexClient(api_key=os.environ["PAGEINDEX_API_KEY"])
submission = pi_client.submit_document(pdf_path)
print(f"pdx index submission: {submission}")
doc_id = submission["doc_id"]
print(f"doc_id: {doc_id}")

pdx index submission: {'doc_id': 'pi-cmrxknla9004j01p5rm0f1mx1'}
doc_id: pi-cmrxknla9004j01p5rm0f1mx1


In [3]:
print(f"pdf path: {pdf_path}")

pdf path: C:\Users\kobyg\AppData\Local\Temp\attention_is_all_you_need.pdf


In [5]:
import time

status = pi_client.get_document(doc_id)["status"]
while status != "completed":
    print(f"status: {status} - waiting...")
    time.sleep(10)
    status = pi_client.get_document(doc_id)["status"]

print("status: completed")

status: completed


### Inspecting the PageIndex tree

The tree is PageIndex's replacement for a vector index: a hierarchy of nodes, each with an id, a title, a starting page (`page_index`), and (optionally) an LLM-generated summary.

In [7]:
from pprint import pprint

tree_response = pi_client.get_tree(doc_id, node_summary=True)
tree = tree_response["result"]
pprint(f"tree: {tree}")

("tree: [{'title': 'Attention Is All You Need', 'node_id': '0000', "
 "'page_index': 1, 'prefix_summary': '# Attention Is All You "
 'Need\\n\\n**Ashish Vaswani***\\nGoogle '
 'Brain\\navaswani@google.com\\n\\n**Noam Shazeer***\\nGoogle '
 'Brain\\nnoam@google.com\\n\\n**Niki Parmar***\\nGoogle '
 'Research\\nnikip@google.com\\n\\n**Jakob Uszkoreit***\\nGoogle '
 'Research\\nusz@google.com\\n\\n**Llion Jones***\\nGoogle '
 'Research\\nllion@google.com\\n\\n**Aidan N. Gomez*** †\\nUniversity of '
 'Toronto\\naidan@cs.toronto.edu\\n\\n**Łukasz Kaiser***\\nGoogle '
 'Brain\\nlukaszkaiser@google.com\\n\\n**Illia Polosukhin*** '
 "‡\\nillia.polosukhin@gmail.com\\n', 'text': '# Attention Is All You "
 'Need\\n\\n**Ashish Vaswani***\\nGoogle '
 'Brain\\navaswani@google.com\\n\\n**Noam Shazeer***\\nGoogle '
 'Brain\\nnoam@google.com\\n\\n**Niki Parmar***\\nGoogle '
 'Research\\nnikip@google.com\\n\\n**Jakob Uszkoreit***\\nGoogle '
 'Research\\nusz@google.com\\n\\n**Llion Jones***\\nGoogle '
 '

### Reasoning over the tree instead of vector search

Instead of embedding the question and searching a vector store, we hand the LLM the tree's titles/summaries (no full page text) and let it pick which node id(s) are relevant. This is a structured-output call, same pattern as `5-structuredoutput.ipynb`.

In [8]:
import json
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class NodeSelection(BaseModel):
    node_ids: list[str] = Field(description="ids of the tree nodes relevant to the question")
    reasoning: str = Field(description="brief justification for the chosen nodes")


model = init_chat_model("gpt-4.1")
model_with_structured_output = model.with_structured_output(NodeSelection)

question = "What is the main architectural contribution of this paper?"

selection_prompt = f"""You are navigating a document's table-of-contents-like tree to answer a question.
Given the tree structure below, return the node id(s) most likely to contain the answer.

Tree:
{json.dumps(tree, indent=2)}

Question: {question}"""

selection = model_with_structured_output.invoke(selection_prompt)
print(f"node selection from tree doc {doc_id}: {selection}")

node selection from tree doc pi-cmrxknla9004j01p5rm0f1mx1: node_ids=['0004'] reasoning="The main architectural contribution is the introduction of the Transformer model, which is described in detail in section 3 'Model Architecture'. This section explains the novel use of self-attention and the absence of recurrence or convolution, which are central to the paper's innovation."


### Fetching content for the selected nodes and answering

With the relevant node id(s) chosen, we pull just those nodes' text from the tree and ask the model to answer using only that context.

In [9]:
def collect_nodes(nodes):
    flat = {}
    for node in nodes:
        flat[node["node_id"]] = node
        if node.get("nodes"):
            flat.update(collect_nodes(node["nodes"]))
    return flat


node_map = collect_nodes(tree)

context_parts = []
for node_id in selection.node_ids:
    node = node_map.get(node_id)
    if node:
        text = node.get("text") or node.get("summary", "")
        context_parts.append(f"[Node {node_id} - {node.get('title')} - page {node.get('page_index')}]\n{text}")

context = "\n\n".join(context_parts)

answer_prompt = f"""Answer the question using only the context below. Cite the node/page you drew from.

Context:
{context}

Question: {question}"""

answer = model.invoke(answer_prompt)
print(answer.content)

The main architectural contribution of this paper is the Transformer model, which uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder in a sequence transduction model, as shown in Figure 1. This departs from previous models by relying entirely on attention mechanisms, specifically self-attention, instead of recurrent or convolutional structures [Node 0004 - 3 Model Architecture - page 2].


No vector store, no embeddings, no chunking — just a document tree and two LLM calls: one to select the relevant section, one to answer from it.